# 3장 2강: SDK 개발 환경 구성과 연결 테스트
## 3. Supabase Client 초기화

In [1]:
import os
from supabase import create_client

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

# Client 초기화
supabase = create_client(url, key)

print(supabase)

## 4. 연결 테스트와 응답·오류 객체 확인

# 3장 3강: SDK 기반 데이터 생성 구현

## 1. SDK insert 구조

In [2]:
# 사용자 추가
response = supabase.table("users").insert({"email": "user03@test.org"}).execute()

response

APIResponse(data=[{'id': '49b370dc-8ee6-46c8-895c-0a67542049ea', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:30:43.916616'}], count=None)

In [3]:
response.data

[{'id': '49b370dc-8ee6-46c8-895c-0a67542049ea',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:30:43.916616'}]

In [7]:
# documents에 추가

response = supabase.table("documents").insert({
    "title":"두 번째 문서",
    "content":"본문 내용",
    "user_id":"389402e1-5390-4f50-a73b-bb45d7181825"

}).execute()

response.data

[{'id': 'd352f7e9-95e5-403d-a98d-0ea3e299d7f3',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'user_id': '389402e1-5390-4f50-a73b-bb45d7181825',
  'status': 'draft',
  'created_at': '2026-09-09T02:37:51.479732+00:00'}]

In [5]:
# 회원 목록 조회
response = supabase.table("users").select("*").execute()

response.data

[{'id': '0cb247c1-86cd-458b-a20f-82b1a4973d58',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:58.928909'},
 {'id': '070cd779-4bca-47a5-8092-bc11915b1a49',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:38.66636'},
 {'id': '3785d4fc-f27a-4e2d-8dae-0547503d7932',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:54:44.444213'},
 {'id': '389402e1-5390-4f50-a73b-bb45d7181825',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:06.836295'},
 {'id': '47434190-c64e-4a1a-857b-162548e53290',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:03:39.645254'},
 {'id': 'b44e27db-5da1-49e4-b4e5-ca1ff93919df',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:03:50.831694'},
 {'id': 'e17f55cc-765f-4340-920e-83b18e148b71',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:11:46.835163'},
 {'id': '49b370dc-8ee6-46c8-895c-0a67542049ea',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:30:43.916616'}]

In [6]:
# single: 레코드 1개, 무조건 1개가 나와야 함. 아니면 예외발생, 변환값은 딕셔너리
response = (
    supabase.table("users")
        .select("*")
        .ilike("email", "user03%")
        .single() 
        .execute()
)


print(response.data)

# id, email, created_at = response.data[0]

# print(f"id:{id}, email:{email}, created_at: {created_at}")

APIError: {'message': 'Cannot coerce the result to a single JSON object', 'code': 'PGRST116', 'hint': None, 'details': 'The result contains 6 rows'}

> limit위치 에러

Why this happens:

maybe_single() means “expect 0 or 1 row”
limit(1) means “return at most 1 row”
they are different stages of the query builder, and limit() is not available after the maybe_single() builder type
So the main fix is: do not chain limit() after maybe_single().

In [10]:
# maybe_single: 레코드 0~1, 0개면 None
response = (
    supabase.table("users")
        .select("*")
        .ilike("email", "user03%")
        .limit(1)
        .offset(1) # 1인덱스에서 1개를 조회
        .maybe_single() 
        .execute()
)

print(response.data)

{'id': '389402e1-5390-4f50-a73b-bb45d7181825', 'email': 'user03@test.org', 'created_at': '2026-09-09T01:55:06.836295'}


In [11]:
response = (
    supabase.table("users")
        .select("*")
        .order("email",desc=True)
        .execute()
)

In [12]:
# LEFT JOIN (default)

response = (
    supabase.table("documents")
        .select("id, title, content, users(id, email)")
        .execute()
)

response.data

[{'id': '18e37b56-232f-4f00-91de-502e8f5a8297',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': '389402e1-5390-4f50-a73b-bb45d7181825',
   'email': 'user03@test.org'}},
 {'id': 'd352f7e9-95e5-403d-a98d-0ea3e299d7f3',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': '389402e1-5390-4f50-a73b-bb45d7181825',
   'email': 'user03@test.org'}}]

In [21]:
# INNER JOIN 

response = (
    supabase.table("documents")
        .select("id, title, content, users!inner(id, email)")
        .ilike("users.email","user%")
        .execute()
)

response.data

[{'id': 'd352f7e9-95e5-403d-a98d-0ea3e299d7f3',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': '389402e1-5390-4f50-a73b-bb45d7181825',
   'email': 'user03@test.org'}},
 {'id': '18e37b56-232f-4f00-91de-502e8f5a8297',
  'title': '첫 번째 문서(수정)',
  'content': '본문 내용',
  'users': {'id': '389402e1-5390-4f50-a73b-bb45d7181825',
   'email': 'user03@test.org'}}]

## 2. 입력값과 DB 컬럼 매핑

## 3. 생성 결과와 오류 응답 처리

In [ ]:
create_document(None, "본문 내용입니다.", 1)

NameError: name 'create_document' is not defined

# 3장 4강: SDK 기반 데이터 조회 구현

## 1. SDK select 구조

## 2. 조건 필터링: eq, order, limit

## 3. 사용자별 조회와 결과 리스트 처리

# 3장 5강: SDK 기반 데이터 수정·삭제 구현

## 1. SDK update 구조

In [15]:
response = (
    supabase.table("documents")
        .select("*")
        .execute()
)

print(response.data)

[{'id': '18e37b56-232f-4f00-91de-502e8f5a8297', 'title': '첫 번째 문서', 'content': '본문 내용', 'user_id': '389402e1-5390-4f50-a73b-bb45d7181825', 'status': 'draft', 'created_at': '2026-09-09T02:11:46.936247+00:00'}, {'id': 'd352f7e9-95e5-403d-a98d-0ea3e299d7f3', 'title': '두 번째 문서', 'content': '본문 내용', 'user_id': '389402e1-5390-4f50-a73b-bb45d7181825', 'status': 'draft', 'created_at': '2026-09-09T02:37:51.479732+00:00'}]


In [17]:
response=(
    supabase.table("documents")
        .update({
            "title":"첫 번째 문서(수정)"
        })
        .eq("id","18e37b56-232f-4f00-91de-502e8f5a8297")
        .execute()
)

response.data

[{'id': '18e37b56-232f-4f00-91de-502e8f5a8297',
  'title': '첫 번째 문서(수정)',
  'content': '본문 내용',
  'user_id': '389402e1-5390-4f50-a73b-bb45d7181825',
  'status': 'draft',
  'created_at': '2026-09-09T02:11:46.936247+00:00'}]

## 2. SDK delete 구조

In [24]:
response = (
    supabase.table("users")
        .select("*")
        .execute()
)

response.data

[{'id': '3785d4fc-f27a-4e2d-8dae-0547503d7932',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:54:44.444213'},
 {'id': '389402e1-5390-4f50-a73b-bb45d7181825',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:06.836295'},
 {'id': '47434190-c64e-4a1a-857b-162548e53290',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:03:39.645254'},
 {'id': 'b44e27db-5da1-49e4-b4e5-ca1ff93919df',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:03:50.831694'},
 {'id': 'e17f55cc-765f-4340-920e-83b18e148b71',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:11:46.835163'},
 {'id': '49b370dc-8ee6-46c8-895c-0a67542049ea',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T02:30:43.916616'}]

In [23]:
response = (
    supabase.table("users")
        .delete()
        .eq("id", '070cd779-4bca-47a5-8092-bc11915b1a49')
        .execute()

)

response.data

[{'id': '070cd779-4bca-47a5-8092-bc11915b1a49',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:38.66636'}]

## 3. 수정·삭제 후 검증과 오류 처리

In [ ]:
# 검증: 상태가 잘 바뀌었는지 다시 조회


# 3장 6강: Supabase Authentication 기본 흐름

## 2. 이메일 기반 회원가입과 로그인

In [ ]:
key = os.getenv("SUPABASE_ANON_KEY")
supabase = create_client(url, key)

print(supabase)

"""
supabase.auth : 인증(로그인) / 인가(접근 제한) 관련 매서드 가지고 있는 객체

테이블명: auth.users

"""

# 회원가입
supabase.auth.sign_up({
    "email":"user01@test.org",
    "password":"password1234",
    "phone":"01011000011"
})

response

APIResponse(data=[{'id': '3785d4fc-f27a-4e2d-8dae-0547503d7932', 'email': 'user03@test.org', 'created_at': '2026-09-09T01:54:44.444213'}, {'id': '389402e1-5390-4f50-a73b-bb45d7181825', 'email': 'user03@test.org', 'created_at': '2026-09-09T01:55:06.836295'}, {'id': '47434190-c64e-4a1a-857b-162548e53290', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:03:39.645254'}, {'id': 'b44e27db-5da1-49e4-b4e5-ca1ff93919df', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:03:50.831694'}, {'id': 'e17f55cc-765f-4340-920e-83b18e148b71', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:11:46.835163'}, {'id': '49b370dc-8ee6-46c8-895c-0a67542049ea', 'email': 'user03@test.org', 'created_at': '2026-09-09T02:30:43.916616'}], count=None)

In [34]:

# 로그인

response = supabase.auth.sign_in_with_password({
    "email": "user01@test.org",
    "password":"password1234"

})

response.user

User(id='95bc3087-c588-4107-81ef-ee595e0f407c', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '95bc3087-c588-4107-81ef-ee595e0f407c'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 40, 16, 795180, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 16, 822400, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 16, 822400, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 9, 3, 52, 24, 70754, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 9, 3, 52, 24, 82899, tzinfo=TzInfo(0)), identities=[UserIdentity(id='95bc3087-c588-4107-81ef-ee595e0f407c', identity_id='af88c3fb-0827-4e

## 3. 세션, 로그아웃, user 객체

In [37]:
# 현재 로그인된 사용자 확인
user = supabase.auth.get_user() #현재 로그인한 사용자 정보, 있으면 로그인 상태, None이면 미 인증 상태

if user: #로그인상태
    print("로그인 상태:", user)
else:
    print("미 로그인 상태")


미 로그인 상태


In [36]:
# 로그아웃
supabase.auth.sign_out()

## 4. 사용자 ID와 테이블 데이터 연결

# 3장 7강: 사용자별 데이터 접근과 권한 관리

## 2. user_id 기반 필터링

In [ ]:


# 본인 문서만 조회


# 3장 8강: SDK 기반 미니 과제 - 사용자별 문서 CRUD

## 2. Auth와 user_id 연결

In [13]:
import os
from supabase import create_client
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_ANON_KEY")

supabase = create_client(url, key)
# print(supabase)
supabase.auth.sign_in_with_password({
    "email":"user01@test.org",
    "password":"password1234"
})

user = supabase.auth.get_user().user
#print(user)
user_id = user.id

# 생성: 로그인 사용자의 문서
response = (
    supabase.table("documents")
        .insert({
            "title":"첫번째 문서",
            "content":"본문 내용",
            "user_id": user_id
        })
        .execute()
)

print(response)

# 조회: 본인 문서만


data=[{'id': 'abe099a9-f87a-47fc-8180-736dd6720404', 'title': '첫번째 문서', 'content': '본문 내용', 'status': 'draft', 'created_at': '2026-09-10T02:10:37.833472+00:00', 'user_id': '95bc3087-c588-4107-81ef-ee595e0f407c'}] count=None


In [ ]:
# CRUD 클래스
class AIlog:

    def __init__(self):
        user = supabase.auth.get_user()
        if not user: #미로그인 상태
            raise PermissionError("로그인이 필요합니다.")

        self.user = user.user

    def create_documents(self, title, content):

        response = (
            supabase.table("documents")
                .insert({
                    "title": title,
                    "content": content,
                    "user_id" : self.user.id
                })
                .execute()
        )

        return response.data
    
    def get_my_documents(self):
        response = (
            supabase.table("documents")
                .select("*, auth.users(email)")
                .eq("user_id", self.user.id)
                .execute()

        )

        return response.data

In [ ]:
ai_log = AIlog()
ai_log.create_documents("세번째 문서", "본문 내용")

[{'id': '736df895-7e16-4f83-bae5-4c4f674030c8',
  'title': '두번째 문서',
  'content': '본문 내용',
  'status': 'draft',
  'created_at': '2026-09-10T01:21:33.886693+00:00',
  'user_id': '95bc3087-c588-4107-81ef-ee595e0f407c'}]

In [ ]:
ai_log.get_my_documents()

In [14]:
import os
from supabase import create_client
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_ANON_KEY")

supabase = create_client(url, key)
# print(supabase)

In [15]:
supabase.auth.sign_up({
    "email":"user02@test.org",
    "password":"password1234"
})

AuthResponse(user=User(id='3478f64b-a064-474f-8cb2-91e2a3966819', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user02@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '3478f64b-a064-474f-8cb2-91e2a3966819'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user02@test.org', phone='', created_at=datetime.datetime(2026, 9, 10, 2, 12, 35, 715978, tzinfo=TzInfo(0)), confirmed_at=None, email_confirmed_at=datetime.datetime(2026, 9, 10, 2, 12, 35, 730545, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 2, 12, 35, 734553, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 2, 12, 35, 738268, tzinfo=TzInfo(0)), identities=[UserIdentity(id='3478f64b-a064-474f-8cb2-91e2a3966819', identity_id='8b00aa87-44c5-4d18-b43c-3e455cfd8ce4', user_id='3478f6

In [20]:
supabase.auth.sign_in_with_password({
    "email": "user01@test.org",
    "password": "password1234"
})

AuthResponse(user=User(id='95bc3087-c588-4107-81ef-ee595e0f407c', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '95bc3087-c588-4107-81ef-ee595e0f407c'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 40, 16, 795180, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 16, 822400, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 16, 822400, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 2, 15, 48, 875146, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 2, 15, 48, 878047, tzinfo=TzInfo(0)), identities=[UserIdentity(id='95bc3087-c588-4107-81ef-ee595e0f407c', identit

In [22]:
# user02로 로그인했을때 결과 없음. 게시글 없음. 신규 가입.
# user01로 로그인했을때 결과 잇음. 기존 회원.
response = (
    supabase.table("documents")
        .select("*")
        .execute()
)

response.data

[{'id': 'df0a7a22-3a89-400e-a7b5-e3efdfdb97d4',
  'title': '첫번째 문서',
  'content': '본문 내용',
  'status': 'draft',
  'created_at': '2026-09-10T01:09:34.585964+00:00',
  'user_id': '95bc3087-c588-4107-81ef-ee595e0f407c'},
 {'id': '736df895-7e16-4f83-bae5-4c4f674030c8',
  'title': '두번째 문서',
  'content': '본문 내용',
  'status': 'draft',
  'created_at': '2026-09-10T01:21:33.886693+00:00',
  'user_id': '95bc3087-c588-4107-81ef-ee595e0f407c'},
 {'id': 'abe099a9-f87a-47fc-8180-736dd6720404',
  'title': '첫번째 문서',
  'content': '본문 내용',
  'status': 'draft',
  'created_at': '2026-09-10T02:10:37.833472+00:00',
  'user_id': '95bc3087-c588-4107-81ef-ee595e0f407c'}]

## 3. 응답·오류 처리와 코드 구조 정리